# Day 10 — Capstone: End-to-End Data Pipeline with Kestra

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-10-capstone-pipeline.ipynb#scrollTo=ca01bb02)

**Course:** Kestra for Data Engineers  
**Badge:** Exam  

You've learned Kestra's building blocks: flows, tasks, triggers, inputs/outputs, error handling, subflows, secrets, and Python integration. Today you'll combine everything into a production-grade data pipeline.

**The pipeline you'll design and validate:**

```
Schedule (daily 7am)
    ↓
ingest   → io.kestra.plugin.core.http.Download (public CSV)
    ↓         retry: maxAttempts 3, exponential backoff
validate → Python: check columns, row count, null percentage
    ↓         Kestra.outputs({valid, row_count, null_pct})
transform → Python: aggregate by group, compute KPIs
    ↓         outputFiles: ['summary.csv']
export   → io.kestra.plugin.core.storage.LocalFiles (save to disk)
    ↓
notify   → io.kestra.plugin.core.log.Log (completion summary)

errors:  → log_failure (error handler on any failure)
```

**By the end of this notebook you will:**
- Build a fully working Kestra YAML pipeline end-to-end using PyYAML
- Validate each pipeline stage locally using real public data
- Add scheduling, inputs, retries, error handling, and quality checks
- Review Kestra best practices before declaring completion

In [ ]:
%pip install -q pyyaml pandas requests

## 1. Pipeline Design — Architecture Overview

Before writing YAML, design the pipeline on paper:
1. **What data?** Public air travel CSV from FSU — monthly passenger counts 1949–1960
2. **What schedule?** Daily at 7am — idempotent, safe to re-run
3. **What validation?** Column names, minimum row count, max null percentage
4. **What transformation?** Yearly totals, peak month, YoY growth rates
5. **What output?** Summary CSV exported to local path
6. **What error handling?** Retry ingestion 3×, flow-level error log
7. **What inputs?** `source_url` and `min_rows` — allows different datasets without code change

In [ ]:
import yaml, json, pandas as pd, requests

# Dataset: FSU public CSV — safe, no auth, stable URL
SOURCE_URL = "https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv"

resp = requests.get(SOURCE_URL)
print(f"HTTP {resp.status_code} — content-length: {len(resp.content)} bytes")

from io import StringIO
df_raw = pd.read_csv(StringIO(resp.text))
df_raw.columns = [c.strip() for c in df_raw.columns]
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(3).to_string())

## 2. Stage 1 — Ingestion: HTTP Download with Retry

In [ ]:
# Stage 1: ingest task definition
ingest_task = {
    "id": "ingest",
    "type": "io.kestra.plugin.core.http.Download",
    "uri": "{{ inputs.source_url }}",
    "retry": {
        "type": "exponential",
        "maxAttempts": 3,
        "delay": "PT5S",
        "multiplier": 2.0,
        "maxDelay": "PT30S"
    },
    "timeout": "PT60S"
}

print("=== STAGE 1: INGEST ===")
print(yaml.dump(ingest_task, default_flow_style=False, sort_keys=False))

# Local simulation
resp = requests.get(SOURCE_URL, timeout=60)
assert resp.status_code == 200, f"Unexpected status: {resp.status_code}"
raw_bytes = resp.content
print(f"Simulation: downloaded {len(raw_bytes):,} bytes")

## 3. Stage 2 — Validation: Schema and Quality Checks

In [ ]:
validate_script = (
    "import pandas as pd, os, sys\n"
    "min_rows = int(os.environ.get('MIN_ROWS', '10'))\n"
    "df = pd.read_csv('raw.csv')\n"
    "df.columns = [c.strip() for c in df.columns]\n"
    "\n"
    "# Column check\n"
    "required_cols = ['Month']\n"
    "missing = [c for c in required_cols if c not in df.columns]\n"
    "if missing:\n"
    "    raise ValueError(f'Missing columns: {missing}')\n"
    "\n"
    "# Row count check\n"
    "if len(df) < min_rows:\n"
    "    raise ValueError(f'Only {len(df)} rows, need {min_rows}')\n"
    "\n"
    "# Null percentage check\n"
    "null_pct = float(df.isnull().mean().max())\n"
    "if null_pct > 0.5:\n"
    "    raise ValueError(f'Null percentage too high: {null_pct:.1%}')\n"
    "\n"
    "print(f'Validation passed: {len(df)} rows, max_null={null_pct:.2%}')\n"
    "Kestra.outputs({\n"
    "    'valid': True,\n"
    "    'row_count': int(len(df)),\n"
    "    'col_count': int(len(df.columns)),\n"
    "    'null_pct': round(null_pct, 4)\n"
    "})\n"
)

validate_task = {
    "id": "validate",
    "type": "io.kestra.plugin.scripts.python.Commands",
    "beforeCommands": ["pip install pandas -q"],
    "inputFiles": {"raw.csv": "{{ outputs.ingest.uri }}"},
    "env": {"MIN_ROWS": "{{ inputs.min_rows }}"},
    "script": validate_script,
    "timeout": "PT2M"
}

print("=== STAGE 2: VALIDATE ===")
print(yaml.dump(validate_task, default_flow_style=False, sort_keys=False))

# Local simulation
from io import StringIO
df = pd.read_csv(StringIO(resp.text))
df.columns = [c.strip() for c in df.columns]
min_rows = 10
missing = [c for c in ['Month'] if c not in df.columns]
assert not missing, f"Missing columns: {missing}"
assert len(df) >= min_rows
null_pct = float(df.isnull().mean().max())
validate_outputs = {'valid': True, 'row_count': len(df), 'col_count': len(df.columns), 'null_pct': round(null_pct, 4)}
print(f"Simulation: {validate_outputs}")

## 4. Stage 3 — Transformation: Aggregations and KPIs

In [ ]:
transform_script = (
    "import pandas as pd\n"
    "df = pd.read_csv('raw.csv')\n"
    "df.columns = [c.strip() for c in df.columns]\n"
    "\n"
    "# Melt wide → long\n"
    "year_cols = [c for c in df.columns if c != 'Month']\n"
    "df_long = df.melt(id_vars=['Month'], var_name='Year', value_name='Passengers')\n"
    "\n"
    "# Yearly aggregations\n"
    "by_year = df_long.groupby('Year')['Passengers'].agg(\n"
    "    total='sum', avg='mean', peak='max'\n"
    ").reset_index().sort_values('Year')\n"
    "\n"
    "# YoY growth rate\n"
    "by_year['yoy_growth_pct'] = by_year['total'].pct_change().mul(100).round(2)\n"
    "by_year['avg'] = by_year['avg'].round(1)\n"
    "\n"
    "# Peak month across all years\n"
    "peak_row = df_long.loc[df_long['Passengers'].idxmax()]\n"
    "peak_month = str(peak_row['Month'])\n"
    "peak_year = str(peak_row['Year'])\n"
    "peak_count = int(peak_row['Passengers'])\n"
    "\n"
    "by_year.to_csv('summary.csv', index=False)\n"
    "print(by_year.to_string(index=False))\n"
    "Kestra.outputs({\n"
    "    'years_processed': int(len(by_year)),\n"
    "    'peak_month': peak_month,\n"
    "    'peak_year': peak_year,\n"
    "    'peak_passengers': peak_count\n"
    "})\n"
)

transform_task = {
    "id": "transform",
    "type": "io.kestra.plugin.scripts.python.Commands",
    "beforeCommands": ["pip install pandas -q"],
    "inputFiles": {"raw.csv": "{{ outputs.ingest.uri }}"},
    "script": transform_script,
    "outputFiles": ["summary.csv"],
    "timeout": "PT5M"
}

print("=== STAGE 3: TRANSFORM ===")
print(yaml.dump(transform_task, default_flow_style=False, sort_keys=False))

# Local simulation
df_long = df.melt(id_vars=['Month'], var_name='Year', value_name='Passengers')
by_year = df_long.groupby('Year')['Passengers'].agg(total='sum', avg='mean', peak='max').reset_index().sort_values('Year')
by_year['yoy_growth_pct'] = by_year['total'].pct_change().mul(100).round(2)
by_year['avg'] = by_year['avg'].round(1)
peak_row = df_long.loc[df_long['Passengers'].idxmax()]
transform_outputs = {
    'years_processed': len(by_year),
    'peak_month': str(peak_row['Month']),
    'peak_year': str(peak_row['Year']),
    'peak_passengers': int(peak_row['Passengers'])
}
print("\nSimulation results:")
print(by_year.to_string(index=False))
print(f"\ntransform_outputs: {transform_outputs}")

## 5. Stage 4 — Export: Save Summary to Disk

In [ ]:
export_task = {
    "id": "export",
    "type": "io.kestra.plugin.core.storage.LocalFiles",
    "inputs": {
        "summary_{{ inputs.report_date | date('yyyyMMdd') }}.csv":
            "{{ outputs.transform.outputFiles['summary.csv'] }}"
    },
    "outputDirectory": "/var/kestra/outputs/airtravel/"
}

notify_task = {
    "id": "notify_success",
    "type": "io.kestra.plugin.core.log.Log",
    "message": (
        "Pipeline complete: "
        "{{ outputs.validate.vars.row_count }} rows ingested, "
        "{{ outputs.transform.vars.years_processed }} years aggregated, "
        "peak={{ outputs.transform.vars.peak_passengers }} passengers "
        "({{ outputs.transform.vars.peak_month }} {{ outputs.transform.vars.peak_year }})"
    )
}

print("=== STAGE 4: EXPORT ===")
print(yaml.dump(export_task, default_flow_style=False, sort_keys=False))
print("=== NOTIFY ===")
print(yaml.dump(notify_task, default_flow_style=False, sort_keys=False))

# Simulate notification message
notify_rendered = (
    f"Pipeline complete: {validate_outputs['row_count']} rows ingested, "
    f"{transform_outputs['years_processed']} years aggregated, "
    f"peak={transform_outputs['peak_passengers']} passengers "
    f"({transform_outputs['peak_month']} {transform_outputs['peak_year']})"
)
print(f"Rendered notification: {notify_rendered}")

## 6. Error Handler — Failure Alerting

In [ ]:
error_handler = [
    {
        "id": "log_failure",
        "type": "io.kestra.plugin.core.log.Log",
        "level": "ERROR",
        "message": (
            "PIPELINE FAILED: {{ flow.id }} | execution={{ execution.id }} | "
            "started={{ execution.startDate }} | "
            "check Kestra UI for task-level error details"
        )
    }
]

print("=== ERROR HANDLER ===")
print(yaml.dump(error_handler, default_flow_style=False, sort_keys=False))

## 7. Schedule Trigger — Daily at 7am

In [ ]:
schedule_trigger = [
    {
        "id": "daily_schedule",
        "type": "io.kestra.plugin.core.trigger.Schedule",
        "cron": "0 7 * * *",
        "timezone": "UTC",
        "backfill": {
            "start": "2026-01-01T07:00:00Z"
        }
    }
]

print("=== SCHEDULE TRIGGER ===")
print(yaml.dump(schedule_trigger, default_flow_style=False, sort_keys=False))

# Cron expression explainer
cron_expr = "0 7 * * *"
fields = cron_expr.split()
labels = ["minute", "hour", "day-of-month", "month", "day-of-week"]
print("Cron breakdown:")
for f, l in zip(fields, labels):
    meaning = f"= {f}" if f != '*' else "= every"
    print(f"  {f:<5} ({l}) {meaning}")
print("→ Runs every day at 07:00 UTC")

## 8. Complete Capstone Flow — Assembled

In [ ]:
capstone_flow = {
    "id": "airtravel-daily-pipeline",
    "namespace": "production.airtravel",
    "description": "Daily air travel data pipeline: ingest → validate → transform → export",
    "labels": {
        "team": "data-engineering",
        "domain": "transport",
        "cadence": "daily"
    },
    "inputs": [
        {
            "id": "source_url",
            "type": "STRING",
            "defaults": "https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv",
            "description": "Source CSV URL — override for different datasets"
        },
        {
            "id": "min_rows",
            "type": "INT",
            "defaults": 10,
            "description": "Minimum row count for validation to pass"
        },
        {
            "id": "report_date",
            "type": "STRING",
            "defaults": "{{ now() | date('yyyy-MM-dd') }}",
            "description": "Report date used in output filename"
        }
    ],
    "triggers": schedule_trigger,
    "tasks": [
        ingest_task,
        validate_task,
        transform_task,
        export_task,
        notify_task
    ],
    "errors": error_handler
}

capstone_yaml = yaml.dump(capstone_flow, default_flow_style=False, sort_keys=False)
print(capstone_yaml)
print(f"--- Total YAML lines: {len(capstone_yaml.splitlines())} ---")

## 9. End-to-End Local Validation

In [ ]:
import requests, pandas as pd
from io import StringIO

print("=" * 60)
print("CAPSTONE PIPELINE — LOCAL END-TO-END SIMULATION")
print("=" * 60)

# STAGE 1: INGEST
print("\n[Stage 1] Ingesting data...")
resp = requests.get("https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv", timeout=30)
assert resp.status_code == 200
print(f"  HTTP {resp.status_code} — {len(resp.content):,} bytes downloaded")

# STAGE 2: VALIDATE
print("\n[Stage 2] Validating schema and quality...")
df = pd.read_csv(StringIO(resp.text))
df.columns = [c.strip() for c in df.columns]
assert 'Month' in df.columns, "Missing Month column"
assert len(df) >= 10, f"Too few rows: {len(df)}"
null_pct = float(df.isnull().mean().max())
assert null_pct <= 0.5, f"Null pct too high: {null_pct:.1%}"
print(f"  Rows: {len(df)} | Cols: {len(df.columns)} | Max null%: {null_pct:.2%}")
print(f"  Validation: PASSED")

# STAGE 3: TRANSFORM
print("\n[Stage 3] Transforming and aggregating...")
df_long = df.melt(id_vars=['Month'], var_name='Year', value_name='Passengers')
by_year = df_long.groupby('Year')['Passengers'].agg(total='sum', avg='mean', peak='max').reset_index().sort_values('Year')
by_year['yoy_growth_pct'] = by_year['total'].pct_change().mul(100).round(2)
by_year['avg'] = by_year['avg'].round(1)
peak_row = df_long.loc[df_long['Passengers'].idxmax()]
print(by_year.to_string(index=False))
print(f"  Peak: {int(peak_row['Passengers'])} passengers in {peak_row['Month']} {peak_row['Year']}")

# STAGE 4: EXPORT (simulated)
print("\n[Stage 4] Exporting summary...")
summary_csv = by_year.to_csv(index=False)
print(f"  Written {len(summary_csv)} bytes to summary.csv")

# NOTIFY
print("\n[Notify] Pipeline complete:")
print(f"  {len(df)} rows ingested, {len(by_year)} years aggregated")
print(f"  Peak: {int(peak_row['Passengers'])} passengers ({peak_row['Month']} {peak_row['Year']})")
print("\n" + "=" * 60)
print("SIMULATION: ALL STAGES PASSED")
print("=" * 60)

## 10. Best Practices Review

In [ ]:
best_practices = [
    ("Idempotency", "Use dated outputs (summary_20260610.csv) so re-runs don't overwrite data"),
    ("Parameterisation", "source_url and min_rows as inputs — no hardcoded values"),
    ("Retries", "Exponential backoff on ingest — handles transient HTTP 429/503"),
    ("Validation", "Fail fast at validate stage before expensive transform"),
    ("Error handling", "errors: block ensures alerting even on unexpected failures"),
    ("Timeouts", "Each task has a timeout — prevents zombie executions"),
    ("Secrets", "Use {{ secret('KEY') }} for credentials — never hardcode in YAML"),
    ("Namespaces", "production.airtravel is hierarchical — group by domain + team"),
    ("Labels", "team + domain + cadence labels enable filtering in the Kestra UI"),
    ("Kestra.outputs", "Publish scalar KPIs as outputs — don't pass large data through vars"),
    ("outputFiles", "Pass large files via internal storage URIs, not environment variables"),
    ("Schedule backfill", "backfill.start lets you catch up missed runs after deploying a new flow")
]

print(f"{'Practice':<20} Guideline")
print("-" * 80)
for practice, guideline in best_practices:
    print(f"{practice:<20} {guideline}")

## 11. Capstone Checklist

In [ ]:
checklist = [
    ("ingest",     "io.kestra.plugin.core.http.Download", True),
    ("retry",      "maxAttempts: 3, exponential backoff", True),
    ("validate",   "column check + row count + null pct + Kestra.outputs", True),
    ("transform",  "yearly agg + YoY growth + peak month + outputFiles", True),
    ("export",     "LocalFiles to /var/kestra/outputs/", True),
    ("notify",     "Log task with rendered output values", True),
    ("errors:",    "log_failure error handler", True),
    ("schedule",   "Daily cron 0 7 * * * with backfill", True),
    ("inputs",     "source_url, min_rows, report_date with defaults", True),
    ("labels",     "team + domain + cadence", True),
    ("timeouts",   "All tasks have timeout set", True),
    ("no secrets", "No hardcoded credentials or webhook URLs", True),
]

all_pass = all(done for _, _, done in checklist)

print("CAPSTONE CHECKLIST")
print("=" * 55)
for component, desc, done in checklist:
    mark = "✓" if done else "✗"
    print(f"  [{mark}] {component:<15} {desc}")
print("=" * 55)
print(f"  {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")

## Challenge — Extend the Pipeline

Add two enhancements to the capstone flow:

**Enhancement 1 — Subflow for reusable validation:**
- Extract the validation logic into a separate flow `production.shared/validate-csv`
- Call it from the main pipeline using `io.kestra.plugin.core.flow.Subflow`
- Pass `source_uri` and `min_rows` as subflow inputs

**Enhancement 2 — Webhook trigger for on-demand runs:**
- Add a second trigger of type `io.kestra.plugin.core.trigger.Webhook`
- The webhook should accept a POST body with `{"source_url": "..."}` and override the default input
- Use `{{ trigger.body.source_url | default(inputs.source_url) }}` to handle both trigger types

In [ ]:
# Enhancement 1: Subflow definition
validate_subflow = {
    "id": "validate-csv",
    "namespace": "production.shared",
    "description": "Reusable CSV validation subflow",
    "inputs": [
        {"id": "source_uri", "type": "STRING"},
        {"id": "min_rows", "type": "INT", "defaults": 10}
    ],
    "tasks": [
        # Your validate task from Stage 2 goes here
        # Hint: replace 'inputFiles: raw.csv: {{ outputs.ingest.uri }}'
        # with  'inputFiles: raw.csv: {{ inputs.source_uri }}'
    ]
}

# Enhancement 2: Webhook trigger
webhook_trigger = {
    "id": "on_demand_webhook",
    "type": "io.kestra.plugin.core.trigger.Webhook",
    "key": "airtravel-trigger-secret-key"  # used in POST URL
}

print("Validate subflow skeleton:")
print(yaml.dump(validate_subflow, default_flow_style=False, sort_keys=False))
print("Webhook trigger:")
print(yaml.dump(webhook_trigger, default_flow_style=False, sort_keys=False))

## Course Recap — Kestra for Data Engineers

| Day | Topic | Key skill |
|-----|-------|----------|
| 1 | Core Concepts | Flows, tasks, triggers, executions, namespaces |
| 2 | First Flow | YAML structure, task chaining, topology view |
| 3 | Tasks & Plugins | Log, Shell, HTTP, Python — task output variables |
| 4 | Triggers | Schedule (cron), Webhook, Flow trigger |
| 5 | Inputs & Outputs | Typed inputs, `Kestra.outputs()`, internal storage |
| 6 | Error Handling | Retry, timeout, `errors:` block, `allowFailure` |
| 7 | Subflows | Modular design, namespace files, reusable flows |
| 8 | Secrets & Variables | `secret()`, namespace vars, Pebble templates |
| 9 | Python Scripts | PROCESS/DOCKER runners, `inputFiles`, chaining |
| 10 | Capstone | Full pipeline: ingest → validate → transform → export |

**Tip:** A complete Kestra pipeline is just a YAML file. Use namespace variables for environment-specific config, secrets for credentials, and subflows for reusable stages — and the whole thing fits in a git repo.

Congratulations on completing the **Kestra for Data Engineers** course!